# Fraise Tokenizer Visualization

Inspect how the saved 8k base tokenizer splits text into tokens.

This notebook assumes your tokenizer module exposes:

```python
load_tokenizer(dataset=None)
```


In [ ]:
from html import escape
from IPython.display import HTML, display

import os
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "tokenizer"
        ).is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not find the Fraise project root from the notebook's working directory."
    )


PROJECT_ROOT = _find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)

from tokenizer import load_tokenizer

tokenizer = load_tokenizer()
print(f"Loaded tokenizer with {tokenizer.get_vocab_size():,} tokens")


FileNotFoundError: No tokenizer found at data/tokenizers/fraise-8k.json and no dataset was supplied for training.

## Visualization helper

Each token is shown as a colored span. The table below it shows the token ID, tokenizer representation, decoded text, and byte length.


In [ ]:
def _visible_text(text: str) -> str:
    """Make whitespace visible while preserving the original decoded token text."""
    return (
        text.replace(" ", "·")
            .replace("\n", "↵\n")
            .replace("\t", "⇥")
    )


def visualize_tokenization(text: str):
    encoding = tokenizer.encode(text)

    decoded_tokens = [
        tokenizer.decode([token_id])
        for token_id in encoding.ids
    ]

    palette = [
        "#fde68a", "#bfdbfe", "#bbf7d0", "#fecaca",
        "#ddd6fe", "#fed7aa", "#bae6fd", "#fbcfe8",
    ]

    spans = []
    for i, (token_id, token_text) in enumerate(zip(encoding.ids, decoded_tokens)):
        color = palette[i % len(palette)]
        label = escape(_visible_text(token_text))
        spans.append(
            f'<span title="id={token_id}" '
            f'style="display:inline-block; background:{color}; color:#111827; '
            f'padding:4px 6px; margin:2px; border-radius:5px; '
            f'font-family:monospace; white-space:pre-wrap;">{label}</span>'
        )

    rows = []
    for i, (token_id, token_repr, token_text) in enumerate(
        zip(encoding.ids, encoding.tokens, decoded_tokens)
    ):
        rows.append(
            "<tr>"
            f"<td>{i}</td>"
            f"<td><code>{token_id}</code></td>"
            f"<td><code>{escape(token_repr)}</code></td>"
            f"<td><code>{escape(_visible_text(token_text))}</code></td>"
            f"<td>{len(token_text.encode('utf-8'))}</td>"
            "</tr>"
        )

    n_chars = len(text)
    n_bytes = len(text.encode("utf-8"))
    n_tokens = len(encoding.ids)

    html = f"""
    <div style="font-family:system-ui,sans-serif; line-height:1.5">
      <div style="margin-bottom:12px">
        <strong>{n_tokens}</strong> tokens &nbsp;·&nbsp;
        <strong>{n_chars}</strong> characters &nbsp;·&nbsp;
        <strong>{n_bytes}</strong> UTF-8 bytes &nbsp;·&nbsp;
        <strong>{n_bytes / max(n_tokens, 1):.2f}</strong> bytes/token
      </div>

      <div style="padding:12px; border:1px solid #d1d5db; border-radius:8px; margin-bottom:16px">
        {''.join(spans)}
      </div>

      <table style="border-collapse:collapse; width:100%; font-size:14px">
        <thead>
          <tr>
            <th style="text-align:left; border-bottom:1px solid #aaa; padding:6px">#</th>
            <th style="text-align:left; border-bottom:1px solid #aaa; padding:6px">ID</th>
            <th style="text-align:left; border-bottom:1px solid #aaa; padding:6px">Tokenizer token</th>
            <th style="text-align:left; border-bottom:1px solid #aaa; padding:6px">Decoded</th>
            <th style="text-align:left; border-bottom:1px solid #aaa; padding:6px">Bytes</th>
          </tr>
        </thead>
        <tbody>{''.join(rows)}</tbody>
      </table>
    </div>
    """

    display(HTML(html))
    return encoding


## Try it

Edit the string and rerun the cell.


In [ ]:
text = "The strawberry problem is interesting because token boundaries hide character structure."

encoding = visualize_tokenization(text)


## Compare multiple strings

Useful for inspecting morphology, whitespace behavior, punctuation, code, and Unicode.


In [ ]:
examples = [
    "strawberry",
    "strawberries",
    "unbelievable",
    "in order to understand this",
    "def hello_world():\n    print('hello')",
    "naïve café",
    "你好，世界",
]

for example in examples:
    print(repr(example))
    visualize_tokenization(example)


## Raw token inspection

Use this when you want the exact IDs or Hugging Face token strings without the HTML visualization.


In [ ]:
text = "A tokenizer should make this easy to inspect."
encoding = tokenizer.encode(text)

print("IDs:")
print(encoding.ids)

print("\nTokenizer representations:")
print(encoding.tokens)

print("\nDecoded one-by-one:")
print([tokenizer.decode([token_id]) for token_id in encoding.ids])
